# Reparameterization Trick Sampler

Chapter 16: Variational Inference and Latent Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Sample z = mu + sigma * epsilon and observe how gradients can flow through mu and sigma.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-16/reparameterization/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's defaults: q = N(mu, sigma^2), n samples, seed
# (same parameters; torch's random draws differ from the page's generator)
mu0, sigma0, n, seed = 0.6, 1.0, 400, 42
torch.manual_seed(seed)

# ── z = mu + sigma * eps with eps ~ N(0, 1): randomness moved into eps ──────
mu = torch.tensor(mu0, dtype=torch.float64, requires_grad=True)
sigma = torch.tensor(sigma0, dtype=torch.float64, requires_grad=True)
eps = torch.randn(n, dtype=torch.float64)
z = mu + sigma * eps
print(f"eps: mean {eps.mean().item():+.4f}, std {eps.std().item():.4f}")
print(f"z:   mean {z.mean().item():+.4f} (mu = {mu0}), std {z.std().item():.4f} (sigma = {sigma0})")

# ── Gradients flow through the sample: d E[z^2] / d mu, d sigma ─────────────
# E[z^2] = mu^2 + sigma^2, so the exact gradients are 2 mu and 2 sigma
loss = (z ** 2).mean()
loss.backward()
print(f"\nreparameterized gradient: d/dmu = {mu.grad.item():.4f} (2 mu = {2 * mu0}), "
      f"d/dsigma = {sigma.grad.item():.4f} (2 sigma = {2 * sigma0})")

# A sample from .sample() is detached: no gradient path back to the parameters
d = torch.distributions.Normal(mu, sigma)
print("rsample keeps the graph:", d.rsample().requires_grad, "| sample does not:", d.sample().requires_grad)

# ── Compare with the score-function (REINFORCE) estimator: same mean, more noise
def grads(estimator, trials=300, m=n):
    out = []
    for _ in range(trials):
        e = torch.randn(m, dtype=torch.float64)
        zz = mu0 + sigma0 * e
        if estimator == "reparam":
            out.append((2 * zz).mean().item())                           # d(z^2)/dmu
        else:
            out.append((zz ** 2 * (zz - mu0) / sigma0 ** 2).mean().item())  # f(z) d log q / dmu
    t = torch.tensor(out)
    return t.mean().item(), t.std().item()


for name in ("reparam", "score"):
    m_, s_ = grads(name)
    print(f"{name:8s} estimate of d/dmu: mean {m_:.4f}, spread {s_:.4f}")

eps: mean +0.0697, std 0.9686
z:   mean +0.6697 (mu = 0.6), std 0.9686 (sigma = 1.0)

reparameterized gradient: d/dmu = 1.3395 (2 mu = 1.2), d/dsigma = 1.9650 (2 sigma = 2.0)
rsample keeps the graph: True | sample does not: False
reparam  estimate of d/dmu: mean 1.2027, spread 0.1052
score    estimate of d/dmu: mean 1.2108, spread 0.2468
